In [1]:
!pip install chronos-forecasting
!pip install transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.preprocessing import StandardScaler

In [3]:
from huggingface_hub import login
login()

In [4]:
df = pd.read_csv("/content/bitola_final_data.csv")
df.head()

,sensorId,timestamp,city,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,16836a55-7140-43e2-9a63-56fac5cba714,2023-12-01 00:00:00+00:00,bitola,58.333333,28.333333,13.666667,943.0,12.00,8.3,47.000000,...,10.00,8.3,8.3,6.98,0.000000,1.000000,-0.5,0.866025,0,1
1,16836a55-7140-43e2-9a63-56fac5cba714,2023-12-01 01:00:00+00:00,bitola,59.250000,13.750000,5.000000,943.0,12.00,9.3,61.694444,...,11.00,9.3,9.3,9.30,0.258819,0.965926,-0.5,0.866025,0,1
2,16836a55-7140-43e2-9a63-56fac5cba714,2023-12-01 02:00:00+00:00,bitola,60.000000,10.000000,4.500000,942.5,12.25,8.3,62.222222,...,11.00,8.3,8.3,8.30,0.500000,0.866025,-0.5,0.866025,0,1
3,16836a55-7140-43e2-9a63-56fac5cba714,2023-12-01 03:00:00+00:00,bitola,59.250000,9.750000,4.500000,942.0,13.00,9.4,61.986111,...,11.00,9.4,9.4,9.40,0.707107,0.707107,-0.5,0.866025,0,1
4,16836a55-7140-43e2-9a63-56fac5cba714,2023-12-01 04:00:00+00:00,bitola,59.500000,5.000000,2.000000,942.0,13.00,9.0,62.208333,...,11.25,9.0,9.0,9.00,0.866025,0.500000,-0.5,0.866025,0,1


In [5]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
count,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,...,152847.000000,152847.000000,152847.000000,152847.000000,152847.000000,1.528470e+05,152847.000000,1.528470e+05,152847.000000,152847.000000
mean,52.811266,27.147240,14.728857,939.616609,16.522118,6.041041,52.123262,52.395130,51.546509,943.443762,...,17.032573,6.209261,6.170199,6.104446,-0.002560,3.292362e-03,-0.066142,2.029823e-02,0.287340,0.422671
std,15.968707,49.876271,26.883015,13.951834,9.077621,3.864767,16.010216,16.348677,14.835662,6.667250,...,8.992836,4.018840,3.983613,3.928963,0.707359,7.068472e-01,0.719120,6.914376e-01,0.452523,0.493986
min,8.750000,0.000000,0.000000,869.000000,-12.000000,0.000000,8.750000,9.333333,9.333333,911.000000,...,-12.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000e+00,-1.000000,-1.000000e+00,0.000000,0.000000
25%,40.750000,5.500000,2.500000,937.250000,9.412879,3.300000,40.250000,40.000000,40.250000,939.684028,...,10.000000,3.300000,3.300000,3.300000,-0.707107,-7.071068e-01,-0.866025,-5.000000e-01,0.000000,0.000000
50%,53.750000,11.250000,6.000000,942.000000,15.750000,5.200000,52.750000,52.879630,52.604167,943.500000,...,16.000000,5.400000,5.400000,5.200000,0.000000,6.120000e-17,0.000000,6.120000e-17,0.000000,0.000000
75%,65.000000,27.000000,14.750000,946.750000,23.250000,7.800000,64.166667,64.750000,63.000000,947.000000,...,23.800000,8.000000,8.000000,7.900000,0.707107,7.071068e-01,0.500000,5.000000e-01,1.000000,1.000000
max,99.000000,1995.000000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,96.500000,971.750000,...,46.750000,32.800000,32.800000,32.800000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000


In [6]:
len(df)

152847

In [7]:
df['sensorId'].value_counts()

,count
sensorId,
874ff9c6-786d-45fc-a90e-48c7ffe03417,16834
87f82783-853b-417d-8964-b5cf11e44873,16688
40f081a6-4095-43f7-bffb-64e2af8c026e,16563
7b316592-8036-41e2-b8dc-b06b6a9afd54,16232
2002,15300
16836a55-7140-43e2-9a63-56fac5cba714,15021
d7060523-b163-4cc3-bc94-970dac72a38a,10931
23b735ef-a996-4a7f-9998-2aa7e78827b0,9538
d241a044-0a06-40c2-9d90-c91fd0a95060,9308


In [8]:
TARGET = 'pm25'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 24

In [9]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [10]:
processed_dfs = []
for sensor_id, group in df.groupby(ID_COL):
    # Challenge 1: Resample to 1-hour frequency to fill gaps/irregularities with NaN
    group = group.sort_values(TIME_COL).set_index(TIME_COL)
    group = group.resample('h').asfreq()

    # Fill missing values for covariates (Weather features)
    # We leave PM10 as NaN if it was missing
    cols_to_fill = [c for c in group.columns if c not in [TARGET, ID_COL,'hour_sin',	'hour_cos',	'month_sin',	'month_cos',	'is_weekend',	'is_heating_season']]
    group[cols_to_fill] = group[cols_to_fill].ffill().bfill()

    group[ID_COL] = sensor_id
    processed_dfs.append(group.reset_index())

full_df = pd.concat(processed_dfs)

In [11]:
full_df

,timestamp,sensorId,city,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,58.333333,28.333333,13.666667,943.0,12.00,8.3,47.000000,...,10.000000,8.3,8.3,6.98,0.000000,1.000000,-0.500000,0.866025,0.0,1.0
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,13.750000,5.000000,943.0,12.00,9.3,61.694444,...,11.000000,9.3,9.3,9.30,0.258819,0.965926,-0.500000,0.866025,0.0,1.0
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,60.000000,10.000000,4.500000,942.5,12.25,8.3,62.222222,...,11.000000,8.3,8.3,8.30,0.500000,0.866025,-0.500000,0.866025,0.0,1.0
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,9.750000,4.500000,942.0,13.00,9.4,61.986111,...,11.000000,9.4,9.4,9.40,0.707107,0.707107,-0.500000,0.866025,0.0,1.0
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.500000,5.000000,2.000000,942.0,13.00,9.0,62.208333,...,11.250000,9.0,9.0,9.00,0.866025,0.500000,-0.500000,0.866025,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11400,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,bitola,61.750000,11.250000,6.000000,942.0,9.00,5.9,63.500000,...,8.277778,5.3,5.9,5.30,-0.965926,0.258819,-0.866025,0.500000,1.0,1.0
11401,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,bitola,63.250000,9.000000,5.666667,942.0,9.00,5.7,63.750000,...,7.777778,5.4,5.7,5.40,-0.866025,0.500000,-0.866025,0.500000,1.0,1.0
11402,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,bitola,63.500000,12.000000,6.000000,942.5,8.25,5.8,63.333333,...,7.685185,5.4,5.8,5.40,-0.707107,0.707107,-0.866025,0.500000,1.0,1.0
11403,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,bitola,63.750000,3.250000,2.500000,943.0,7.75,5.6,63.750000,...,7.222222,5.0,5.6,5.00,-0.500000,0.866025,-0.866025,0.500000,1.0,1.0


In [12]:
full_df = full_df.drop(columns=['pm10','city'],axis=1)

In [13]:
full_df.columns

Index(['timestamp', 'sensorId', 'humidity', 'pm25', 'pressure', 'temperature',
       'wind_speed', 'neighbor1_humidity', 'neighbor2_humidity',
       'neighbor3_humidity', 'neighbor1_pressure', 'neighbor2_pressure',
       'neighbor3_pressure', 'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [14]:
full_df.isnull().sum()

,0
timestamp,0
sensorId,0
humidity,0
pm25,21130
pressure,0
temperature,0
wind_speed,0
neighbor1_humidity,0
neighbor2_humidity,0
neighbor3_humidity,0


In [15]:
def get_hour_sin(timestamp):
 return  np.sin(2 * np.pi * timestamp.dt.hour / 24)

def get_hour_cos(timestamp):
  return  np.cos(2 * np.pi * timestamp.dt.hour / 24)

def get_month_sin(timestamp):
  return np.sin(2 * np.pi * (timestamp.dt.month - 1) / 12)

def get_month_cos(timestamp):
  return np.cos(2 * np.pi * (timestamp.dt.month - 1) / 12)

def get_is_weekend(timestamp):
    return timestamp.dt.dayofweek.isin([5, 6]).astype(int)

def get_is_heating_season(timestamp):
    return timestamp.dt.month.isin([11, 12, 1, 2, 3]).astype(int)

In [16]:
full_df['hour_sin'] = full_df['hour_sin'].fillna(get_hour_sin(full_df['timestamp']))
full_df['hour_cos'] = full_df['hour_cos'].fillna(get_hour_cos(full_df['timestamp']))
full_df['month_sin'] = full_df['month_sin'].fillna(get_month_sin(full_df['timestamp']))
full_df['month_cos'] = full_df['month_cos'].fillna(get_month_cos(full_df['timestamp']))
full_df['is_weekend'] = full_df['is_weekend'].fillna(get_is_weekend(full_df['timestamp']))
full_df['is_heating_season'] = full_df['is_heating_season'].fillna(get_is_heating_season(full_df['timestamp']))

In [17]:
full_df.isnull().sum()

,0
timestamp,0
sensorId,0
humidity,0
pm25,21130
pressure,0
temperature,0
wind_speed,0
neighbor1_humidity,0
neighbor2_humidity,0
neighbor3_humidity,0


In [18]:
full_df.describe()

,humidity,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,neighbor2_pressure,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
count,173977.000000,152847.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,...,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,1.739770e+05,173977.000000,1.739770e+05,173977.000000,173977.000000
mean,52.438657,14.728857,939.305932,16.635632,5.968529,52.402565,52.236401,51.409615,943.364846,942.726616,...,17.105841,6.121637,6.101540,5.997510,-0.000125,1.015216e-04,-0.048883,1.865304e-02,0.286785,0.426689
std,15.878385,26.883015,14.356314,9.118732,3.888406,15.995469,16.154622,14.656721,6.778492,6.736012,...,8.987505,4.040755,3.979501,3.906067,0.707108,7.071095e-01,0.712512,6.997104e-01,0.452262,0.494598
min,8.750000,0.000000,869.000000,-12.000000,0.000000,8.750000,9.333333,9.333333,911.000000,911.000000,...,-12.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000e+00,-1.000000,-1.000000e+00,0.000000,0.000000
25%,40.000000,2.500000,937.000000,9.400000,3.200000,41.000000,39.750000,40.902778,939.000000,938.250000,...,10.000000,3.200000,3.200000,3.200000,-0.707107,-7.071068e-01,-0.866025,-5.000000e-01,0.000000,0.000000
50%,53.000000,6.000000,942.000000,16.000000,5.100000,52.500000,52.250000,52.000000,943.000000,942.666667,...,16.250000,5.200000,5.200000,5.200000,0.000000,6.120000e-17,0.000000,6.120000e-17,0.000000,0.000000
75%,64.250000,14.750000,946.666667,23.500000,7.700000,64.250000,64.500000,62.750000,947.000000,947.000000,...,24.000000,7.900000,7.900000,7.700000,0.707107,7.071068e-01,0.500000,5.000000e-01,1.000000,1.000000
max,99.000000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,96.500000,971.750000,971.750000,...,46.750000,32.800000,32.800000,32.800000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000


In [30]:
percentile = np.nanpercentile(full_df['pm25'], 80)
print("80th percentile of PM25:", percentile)


80th percentile of PM25: 1.5681445233141453


In [20]:
full_df['pm25'] = full_df['pm25'].clip(upper=percentile)

In [21]:
full_df.describe()

,humidity,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,neighbor2_pressure,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
count,173977.000000,152847.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,...,173977.000000,173977.000000,173977.000000,173977.000000,173977.000000,1.739770e+05,173977.000000,1.739770e+05,173977.000000,173977.000000
mean,52.438657,8.191238,939.305932,16.635632,5.968529,52.402565,52.236401,51.409615,943.364846,942.726616,...,17.105841,6.121637,6.101540,5.997510,-0.000125,1.015216e-04,-0.048883,1.865304e-02,0.286785,0.426689
std,15.878385,6.680164,14.356314,9.118732,3.888406,15.995469,16.154622,14.656721,6.778492,6.736012,...,8.987505,4.040755,3.979501,3.906067,0.707108,7.071095e-01,0.712512,6.997104e-01,0.452262,0.494598
min,8.750000,0.000000,869.000000,-12.000000,0.000000,8.750000,9.333333,9.333333,911.000000,911.000000,...,-12.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000e+00,-1.000000,-1.000000e+00,0.000000,0.000000
25%,40.000000,2.500000,937.000000,9.400000,3.200000,41.000000,39.750000,40.902778,939.000000,938.250000,...,10.000000,3.200000,3.200000,3.200000,-0.707107,-7.071068e-01,-0.866025,-5.000000e-01,0.000000,0.000000
50%,53.000000,6.000000,942.000000,16.000000,5.100000,52.500000,52.250000,52.000000,943.000000,942.666667,...,16.250000,5.200000,5.200000,5.200000,0.000000,6.120000e-17,0.000000,6.120000e-17,0.000000,0.000000
75%,64.250000,14.750000,946.666667,23.500000,7.700000,64.250000,64.500000,62.750000,947.000000,947.000000,...,24.000000,7.900000,7.900000,7.700000,0.707107,7.071068e-01,0.500000,5.000000e-01,1.000000,1.000000
max,99.000000,18.666667,971.750000,47.500000,32.800000,99.000000,99.000000,96.500000,971.750000,971.750000,...,46.750000,32.800000,32.800000,32.800000,1.000000,1.000000e+00,1.000000,1.000000e+00,1.000000,1.000000


In [22]:
import joblib
numeric_features = [
    'humidity', 'pressure', 'temperature', 'wind_speed',
    'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
    'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
    'neighbor1_temperature', 'neighbor2_temperature', 'neighbor3_temperature',
    'neighbor1_wind_speed', 'neighbor2_wind_speed', 'neighbor3_wind_speed'
]
scaler = joblib.load("/content/feature_scaler.pkl")
full_df[numeric_features] = scaler.fit_transform(full_df[numeric_features])

In [23]:
scalerpm25 = StandardScaler()
full_df['pm25'] = scalerpm25.fit_transform(full_df[['pm25']])
full_df

,timestamp,sensorId,humidity,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.371240,0.819657,0.257314,-0.508365,0.599597,-0.337757,0.913896,0.654336,...,-0.790638,0.539100,0.552448,0.251530,0.000000,1.000000,-0.500000,0.866025,0.0,1.0
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.428971,-0.477720,0.257314,-0.508365,0.856773,0.580909,-0.220974,1.114876,...,-0.679372,0.786579,0.803736,0.845479,0.258819,0.965926,-0.500000,0.866025,0.0,1.0
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.476205,-0.552569,0.222486,-0.480949,0.599597,0.613904,-0.215815,1.183105,...,-0.679372,0.539100,0.552448,0.589467,0.500000,0.866025,-0.500000,0.866025,0.0,1.0
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.428971,-0.552569,0.187658,-0.398700,0.882490,0.599143,-0.200340,1.222905,...,-0.679372,0.811327,0.828865,0.871081,0.707107,0.707107,-0.500000,0.866025,0.0,1.0
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.444715,-0.926812,0.187658,-0.398700,0.779620,0.613036,-0.200340,1.234276,...,-0.651555,0.712335,0.728350,0.768675,0.866025,0.500000,-0.500000,0.866025,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11400,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.586418,-0.328023,0.187658,-0.837359,-0.017624,0.693788,2.120984,1.171733,...,-0.982262,-0.203338,-0.050645,-0.178572,-0.965926,0.258819,-0.866025,0.500000,1.0,1.0
11401,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.680886,-0.377922,0.187658,-0.837359,-0.069059,0.709418,2.368592,1.222905,...,-1.037895,-0.178590,-0.100903,-0.152970,-0.866025,0.500000,-0.866025,0.500000,1.0,1.0
11402,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.696631,-0.328023,0.222486,-0.919608,-0.043342,0.683369,2.605883,1.297450,...,-1.048198,-0.178590,-0.075774,-0.152970,-0.707107,0.707107,-0.866025,0.500000,1.0,1.0
11403,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.712376,-0.851964,0.257314,-0.974440,-0.094777,0.709418,2.832857,1.329037,...,-1.099710,-0.277582,-0.126031,-0.255375,-0.500000,0.866025,-0.866025,0.500000,1.0,1.0


In [24]:
context_df = full_df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
ground_truth_df = full_df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)

/tmp/ipykernel_1052/1967692071.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  context_df = full_df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
/tmp/ipykernel_1052/1967692071.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ground_truth_df = full_df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)


In [25]:
ground_truth_df

,timestamp,sensorId,humidity,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor3_temperature,neighbor1_wind_speed,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,is_weekend,is_heating_season
0,2025-11-30 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.680886,-0.365447,-0.021310,-0.919608,-0.351952,0.615641,1.100633,0.688450,...,-0.790638,-0.525061,-0.377320,-0.357780,0.000000,1.000000,-0.866025,0.5,1.0,1.0
1,2025-11-30 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.728120,-0.278124,-0.021310,-0.947024,0.548162,0.584382,1.077936,0.620221,...,-0.790638,-0.500313,0.502190,0.538264,0.258819,0.965926,-0.866025,0.5,1.0,1.0
2,2025-11-30 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.570673,-0.552569,0.013518,-0.947024,0.496727,0.537493,1.031510,0.335937,...,-0.716461,-0.302330,0.451932,0.487061,0.500000,0.866025,-0.866025,0.5,1.0,1.0
3,2025-11-30 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.602163,-0.215750,0.048346,-0.947024,0.548162,0.558333,1.079999,0.483765,...,-0.818454,-0.129094,0.502190,0.538264,0.707107,0.707107,-0.866025,0.5,1.0,1.0
4,2025-11-30 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.665142,-0.365447,0.048346,-0.947024,0.548162,0.615641,1.112498,0.608850,...,-0.790638,-0.030103,0.502190,0.538264,0.866025,0.500000,-0.866025,0.5,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.586418,-0.328023,0.187658,-0.837359,-0.017624,0.693788,2.120984,1.171733,...,-0.982262,-0.203338,-0.050645,-0.178572,-0.965926,0.258819,-0.866025,0.5,1.0,1.0
356,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.680886,-0.377922,0.187658,-0.837359,-0.069059,0.709418,2.368592,1.222905,...,-1.037895,-0.178590,-0.100903,-0.152970,-0.866025,0.500000,-0.866025,0.5,1.0,1.0
357,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.696631,-0.328023,0.222486,-0.919608,-0.043342,0.683369,2.605883,1.297450,...,-1.048198,-0.178590,-0.075774,-0.152970,-0.707107,0.707107,-0.866025,0.5,1.0,1.0
358,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.712376,-0.851964,0.257314,-0.974440,-0.094777,0.709418,2.832857,1.329037,...,-1.099710,-0.277582,-0.126031,-0.255375,-0.500000,0.866025,-0.866025,0.5,1.0,1.0


In [26]:
ground_truth_df['timestamp'].max()

Timestamp('2025-11-30 23:00:00+0000', tz='UTC')

In [27]:
ground_truth_df['sensorId'].value_counts()

,count
sensorId,
16836a55-7140-43e2-9a63-56fac5cba714,24
2001,24
2002,24
23b735ef-a996-4a7f-9998-2aa7e78827b0,24
2819ecbb-5de3-4092-aa1d-4ba3a8c40add,24
30dab8a6-ff63-43ce-9a3b-99f1f3f7054d,24
40f081a6-4095-43f7-bffb-64e2af8c026e,24
7b316592-8036-41e2-b8dc-b06b6a9afd54,24
874ff9c6-786d-45fc-a90e-48c7ffe03417,24


In [28]:
print("Loading Chronos-2 and generating forecasts...")
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="auto",
    dtype=torch.bfloat16,
)

Loading Chronos-2 and generating forecasts...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

In [29]:
future_df = ground_truth_df.drop(columns='pm25',axis=1)

In [31]:
forecast_df = pipeline.predict_df(
    df=context_df,
    prediction_length=PREDICTION_LENGTH,
    target=TARGET,
    id_column=ID_COL,
    future_df = future_df
)

In [32]:
# Merge predictions with actual values to align them
eval_df = pd.merge(
    forecast_df[[ID_COL, TIME_COL, 'predictions']],
    ground_truth_df[[ID_COL, TIME_COL, TARGET]],
    on=[ID_COL, TIME_COL]
)

In [33]:
eval_df.columns

Index(['sensorId', 'timestamp', 'predictions', 'pm25'], dtype='object')

In [34]:
eval_df

,sensorId,timestamp,predictions,pm25
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 00:00:00+00:00,-0.400391,-0.365447
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 01:00:00+00:00,-0.507812,-0.278124
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 02:00:00+00:00,-0.558594,-0.552569
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 03:00:00+00:00,-0.574219,-0.215750
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 04:00:00+00:00,-0.511719,-0.365447
...,...,...,...,...
355,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,-0.847656,-0.328023
356,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,-0.917969,-0.377922
357,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,-0.980469,-0.328023
358,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,-1.023438,-0.851964


In [36]:
eval_df['predictions'] = scalerpm25.inverse_transform(eval_df[['predictions']])
eval_df['pm25'] = scalerpm25.inverse_transform(eval_df[['pm25']])

In [37]:
eval_df

,sensorId,timestamp,predictions,pm25
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 00:00:00+00:00,5.516572,5.750000
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 01:00:00+00:00,4.798979,6.333333
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 02:00:00+00:00,4.459753,4.500000
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 03:00:00+00:00,4.355375,6.750000
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-30 04:00:00+00:00,4.772884,5.750000
...,...,...,...,...
355,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,2.528774,6.000000
356,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,2.059077,5.666667
357,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,1.641568,6.000000
358,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,1.354530,2.500000


In [39]:
eval_df.dropna(axis=0,inplace=True)

In [40]:
eval_df.isnull().sum()

,0
sensorId,0
timestamp,0
predictions,0
pm25,0


In [41]:
y_true = eval_df['pm25']
y_pred = eval_df['predictions']

In [42]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- Global Model Evaluation ---")
print(f"RMSE: {rmse:.4f}")
print(f"R2 score: {r2:.4f}")

--- Global Model Evaluation ---
RMSE: 4.1249
R2 score: 0.5704


In [44]:
pm25_scaler = scalerpm25
joblib.dump(pm25_scaler, "pm25_scaler.pkl")

['pm25_scaler.pkl']

In [45]:
pipeline.save_pretrained("my_chronos_pipeline")